In [1]:
%matplotlib inline
import matplotlib.pyplot as plt

import seaborn as sns

sns.set_theme()

plt.rcParams["figure.figsize"] = 9, 4.51

# Extractors Tutorial

## Introduction

This tutorial is a guide of how you can create your custom feature extraction routine
and add this extractor to feets.

## Fundamentals

-   The feature extraction as modeled as a class.
-   The class must inerith from `feets.Extractor`
-   The extractor class must has at least two elements
    1. **features**: list with the name of the features that this extractor generates.
    2. **extract**: a method with light-curve data vectos as parameters, that implements
       the feature extraction logic. `extract()` must return a dictionary with keys equal
       to the `features` list.

## Example 1: `MaxMagMinTime` extractor

Let's say we need to create a feature extractor called **MaxMagMinTime** that must return 2 features:

1. **magmax**: The maximun magnitude
2. **mintime**: the minimun time

In [2]:
import feets

class MaxMagMinTime(feets.Extractor):  # must inherit from Extractor
    features = ["magmax", "mintime"]  # The names of the expected features

    # The params are light-curve data vectors
    def extract(self, magnitude, time):
        # The return value must be a dict with the same values
        # defined in  features
        return {"magmax": magnitude.max(), "mintime": time.min()}

Finally to make the extractor available for the `FeaturSpace` class, you need to register it with the command:

In [3]:
from feets import extractor_registry
extractor_registry.register_extractor(MaxMagMinTime)

__main__.MaxMagMinTime

Now the extractor is available as any of the already defined ones in feets:

In [4]:
# let's create the feature-space
fs = feets.FeatureSpace(only={"magmax", "mintime"})
fs

<FeatureSpace: MaxMagMinTime()>

In [5]:
# extract the features
features = fs.extract(time=[1, 2, 3], magnitude=[100, 200, 300])
features.as_frame()

<class 'joblib.parallel.Parallel'>


Features,mintime,magmax
Light Curve,,
0,1,300


## Dependencies

An extractor may also depend on the features computed by other extractors in order to work.
This dependencies must also be specified as parameters in the `extract()` method

## Example 2: `TimeDuration` extractor

Now let's say we need an extractor **TimeDuration** that computes:

1. **duration**: time-series duration. max_time - min_time

We can use the feature **mintime**, computed by **MaxMagMinTime**, to compute this new feature


In [6]:
class TimeDuration(feets.Extractor):
    features = ["duration"] 

    # mintime is deefined as a dependency
    def extract(self, time, mintime):
        return {"duration": time.max() - mintime}

extractor_registry.register_extractor(TimeDuration)

__main__.TimeDuration

Now, whenever we want to compute the **duration**, the `FeatureSpace` will ensure that every dependency (in this case, **mintime**) is previously computed by their respective extractors before executing the **TimeDuration** extractor

In [7]:
# MaxMagMinTime is a dependency for TimeDuration and should
# automatically be added to this FeatureSpace even if we don't
# specify it
fs = feets.FeatureSpace(only={"duration"})
fs

<FeatureSpace: MaxMagMinTime(), TimeDuration()>

In [8]:

features = fs.extract(time=[1, 2, 3], magnitude=[100, 200, 300])
features.as_frame()

<class 'joblib.parallel.Parallel'>


Features,duration
Light Curve,
0,2
